# CRIVO — Escola de Composição Semântica (GPU)

**Objetivo desta fase:** transformar associações conceituais em respostas compostas e naturais:

`conceito → relação → composição → frase`

Este notebook parte do checkpoint **conceitual real** salvo no seu Google Drive. Ele treina somente os blocos superiores do Transformer CRIVO 17,43M, usa respostas-alvo inteiras (não palavras isoladas), inclui replay para reduzir esquecimento e mantém o resultado com `ativo_no_chat=false`.

Nenhum modelo externo/API é usado.


In [ ]:
!nvidia-smi
!pip -q install 'tokenizers>=0.20,<1'

import torch, pathlib, random, json, hashlib, shutil, time, sys, re
from google.colab import drive

assert torch.cuda.is_available(), 'Ative GPU em Ambiente de execução > Alterar tipo de ambiente > GPU.'
DEVICE='cuda'
SEED=202610112319
random.seed(SEED); torch.manual_seed(SEED)

drive.mount('/content/drive')
ROOT=pathlib.Path('/content/drive/MyDrive/CRIVO/checkpoints')
SRCROOT=ROOT/'escola_conceitual_real'
assert SRCROOT.exists(), f'Não encontrei {SRCROOT}'

cands=sorted([p for p in SRCROOT.glob('crivo_conceitual_*') if (p/'pesos.pt').exists()])
assert cands, 'Nenhum checkpoint conceitual final encontrado.'
BASE=cands[-1]
print('✅ GPU:', torch.cuda.get_device_name(0))
print('✅ checkpoint de entrada:', BASE)


In [ ]:
%cd /content
!rm -rf CRIVO
!git clone -q https://github.com/HROSONE/CRIVO.git
sys.path.insert(0,'/content/CRIVO')

from linguagem_profunda import Configuracao, LinguagemProfunda, fonte_dialogo, ESPECIAIS
from tokenizers import Tokenizer

estado=torch.load(BASE/'pesos.pt', map_location='cpu', weights_only=True)
CFG={'vocabulario':8192,'dimensao':384,'camadas':8,'cabecas':8,'contexto':256,'dropout':0.1}
assert estado['config']==CFG

modelo=LinguagemProfunda(Configuracao(**CFG))
modelo.load_state_dict(estado['modelo'], strict=True)
modelo=modelo.to(DEVICE)

tok=Tokenizer.from_file(str(BASE/'tokenizer.json')); tok.encode_special_tokens=True
assert tok.get_vocab_size()==8192
assert sum(p.numel() for p in modelo.parameters())==17_428_224

PAD=tok.token_to_id('<pad>')
USR=tok.token_to_id('<usuario>')
AST=tok.token_to_id('<assistente>')
FIM=tok.token_to_id('<fim>')
DOC=tok.token_to_id('<documento>')
PROIB=[tok.token_to_id(t) for t in ESPECIAIS[:-1]]

print('✅ CRIVO real carregado:', sum(p.numel() for p in modelo.parameters()), 'parâmetros')
print('✅ passo de origem:', estado.get('passo'))


## Currículo

A escola usa conceitos como **copo, telescópio, hipótese, memória e contexto**, mas o alvo não é decorar perguntas. Cada fato aparece em várias operações linguísticas: definição, função, invariância, contraste, consequência e explicação.

Os testes no final usam formulações reservadas que não aparecem no treino.


In [ ]:
CONCEITOS = {
'copo': {
 'categoria':'recipiente',
 'funcoes':['conter líquidos','permitir beber'],
 'variacoes':['vidro','plástico','metal','barro'],
 'frase':'Um copo é um recipiente usado para conter líquidos e beber.'
},
'garrafa': {
 'categoria':'recipiente',
 'funcoes':['armazenar líquidos','transportar bebidas'],
 'variacoes':['vidro','plástico','metal'],
 'frase':'Uma garrafa é um recipiente usado para armazenar e transportar líquidos.'
},
'cadeira': {
 'categoria':'móvel',
 'funcoes':['servir de assento','sustentar uma pessoa sentada'],
 'variacoes':['madeira','metal','plástico'],
 'frase':'Uma cadeira é um móvel feito para uma pessoa se sentar.'
},
'martelo': {
 'categoria':'ferramenta',
 'funcoes':['golpear objetos','cravar pregos'],
 'variacoes':['aço','madeira','fibra'],
 'frase':'Um martelo é uma ferramenta usada para golpear objetos, como ao cravar pregos.'
},
'telescópio': {
 'categoria':'instrumento óptico',
 'funcoes':['coletar luz','observar objetos distantes'],
 'variacoes':['refrator','refletor'],
 'frase':'Um telescópio é um instrumento óptico que coleta luz para observar objetos distantes.'
},
'microscópio': {
 'categoria':'instrumento óptico',
 'funcoes':['ampliar estruturas pequenas','observar detalhes microscópicos'],
 'variacoes':['óptico','eletrônico'],
 'frase':'Um microscópio é um instrumento usado para observar estruturas pequenas em maior detalhe.'
},
'termômetro': {
 'categoria':'instrumento de medição',
 'funcoes':['medir temperatura'],
 'variacoes':['digital','líquido','infravermelho'],
 'frase':'Um termômetro é um instrumento de medição usado para medir temperatura.'
},
'computador': {
 'categoria':'máquina eletrônica',
 'funcoes':['processar informações','executar programas'],
 'variacoes':['portátil','desktop','servidor'],
 'frase':'Um computador é uma máquina eletrônica capaz de processar informações e executar programas.'
},
'água': {
 'categoria':'substância',
 'funcoes':['hidratar organismos','atuar como solvente'],
 'variacoes':['líquida','sólida','gasosa'],
 'frase':'A água é uma substância que pode mudar de estado físico sem deixar de ser água.'
},
'planeta': {
 'categoria':'corpo celeste',
 'funcoes':['orbitar uma estrela'],
 'variacoes':['rochoso','gasoso'],
 'frase':'Um planeta é um corpo celeste que orbita uma estrela e possui gravidade própria.'
},
'hipótese': {
 'categoria':'proposição provisória',
 'funcoes':['representar uma possibilidade a testar'],
 'variacoes':['plausível','improvável'],
 'frase':'Uma hipótese é uma proposição provisória que pode ser testada e não equivale a um fato confirmado.'
},
'contexto': {
 'categoria':'informação circundante',
 'funcoes':['ajudar a interpretar palavras','resolver referências'],
 'variacoes':['linguístico','situacional'],
 'frase':'Contexto é a informação ao redor de uma expressão que ajuda a interpretar seu significado.'
},
'memória': {
 'categoria':'processo de armazenamento e recuperação',
 'funcoes':['reter informação','recuperar informação anterior'],
 'variacoes':['curta duração','longa duração'],
 'frase':'Memória é o processo de reter e recuperar informação.'
},
'correção': {
 'categoria':'atualização de informação',
 'funcoes':['substituir informação anterior errada'],
 'variacoes':['explícita','contextual'],
 'frase':'Uma correção atualiza o estado vigente substituindo uma informação anterior incorreta.'
},
'pergunta': {
 'categoria':'ato de linguagem',
 'funcoes':['solicitar informação','pedir esclarecimento'],
 'variacoes':['direta','indireta'],
 'frase':'Uma pergunta é um ato de linguagem usado para solicitar informação ou esclarecimento.'
},
'resposta': {
 'categoria':'ato de linguagem',
 'funcoes':['fornecer informação solicitada'],
 'variacoes':['curta','detalhada'],
 'frase':'Uma resposta fornece informação ao pedido feito e deve considerar o contexto.'
},
}

TREINO=[]
def add(q,a,tipo):
    TREINO.append({'q':q,'a':a,'tipo':tipo})

for nome,c in CONCEITOS.items():
    cat=c['categoria']; f1=c['funcoes'][0]; frase=c['frase']
    add(f'O que é {nome}?', frase, 'definicao')
    add(f'Explique o significado de {nome} em uma frase.', frase, 'definicao')
    add(f'Em termos simples, como você define {nome}?', frase, 'definicao')
    add(f'A que categoria pertence {nome}?', f'{nome.capitalize()} pertence à categoria {cat}.', 'categoria')
    add(f'Que tipo de coisa é {nome}?', f'{nome.capitalize()} é um tipo de {cat}.', 'categoria')
    add(f'Para que serve {nome}?', f'{nome.capitalize()} pode servir para {f1}.', 'funcao')
    add(f'Qual é uma função de {nome}?', f'Uma função de {nome} é {f1}.', 'funcao')
    if c['variacoes']:
        v=c['variacoes'][0]
        add(f'Se {nome} for {v}, ele deixa de ser {nome}?', f'Não. Essa variação não muda o conceito central: continua sendo {nome}.', 'invariancia')
        add(f'Mudar a forma ou o material faz {nome} deixar de pertencer à categoria {cat}?', f'Não. Ele pode variar na forma ou no material e ainda continuar sendo {nome}, um {cat}.', 'invariancia')
    add(f'Conecte {nome}, {cat} e sua função principal.', f'{nome.capitalize()} é um {cat} cuja função pode ser {f1}.', 'composicao')
    add(f'Faça uma explicação curta juntando categoria e função de {nome}.', f'{nome.capitalize()} é um {cat} e pode {f1}.', 'composicao')

# Regras conceituais compostas
REGRAS=[
 ('Se algo continua com a mesma função e categoria apesar de mudar de material, a mudança de material altera necessariamente o conceito?',
  'Não. Uma mudança de material não altera necessariamente o conceito se a função e a categoria relevantes continuam as mesmas.'),
 ('Se uma hipótese ainda não foi confirmada, ela deve ser tratada como fato?',
  'Não. Uma hipótese não confirmada deve permanecer como possibilidade provisória, não como fato.'),
 ('Quando uma pessoa corrige uma informação anterior, qual informação deve valer depois?',
  'Depois de uma correção explícita, deve valer a informação corrigida, substituindo a anterior.'),
 ('Por que o contexto pode mudar a interpretação de uma palavra?',
  'Porque o contexto fornece informação ao redor da palavra e ajuda a determinar qual sentido ou referência é apropriado.'),
 ('Como memória e contexto podem trabalhar juntos numa conversa?',
  'A memória recupera informação anterior e o contexto ajuda a interpretar como essa informação se aplica ao turno atual.'),
 ('Qual a diferença entre uma pergunta e uma resposta?',
  'A pergunta solicita informação ou esclarecimento; a resposta fornece informação em relação ao pedido.'),
 ('Se eu sei a categoria de um objeto e também sua função, o que consigo fazer?',
  'Você consegue formar uma descrição mais completa, combinando o que o objeto é com o que ele faz.'),
]
for q,a in REGRAS:
    add(q,a,'regra')

# Replay: linguagem simples e relações gerais para reduzir esquecimento
REPLAY=[
 'A Terra orbita o Sol e a Lua orbita a Terra.',
 'O coração bombeia sangue através dos vasos do corpo.',
 'Uma biblioteca organiza livros para que possam ser encontrados novamente.',
 'Uma conclusão deve respeitar as evidências disponíveis.',
 'A pressão atmosférica pode variar com a altitude.',
 'Uma palavra pode assumir sentidos diferentes dependendo do contexto.',
 'Uma explicação clara conecta ideias em uma ordem compreensível.',
 'Uma correção deve substituir a informação errada pela informação atualizada.',
 'Uma pergunta pode indicar que falta alguma informação.',
 'Um objeto pode manter sua identidade mesmo quando algumas propriedades externas mudam.',
 'Uma categoria agrupa entidades que compartilham características relevantes.',
 'Uma função descreve o que algo faz ou para que pode ser usado.',
]

random.shuffle(TREINO)
print('✅ exemplos de composição:',len(TREINO),'| replay:',len(REPLAY))


In [ ]:
def cod(texto):
    return tok.encode(texto, add_special_tokens=False).ids

def exemplo_chat(q,a):
    fonte=[USR]+cod(q)+[FIM,AST]
    resp=cod(a)+[FIM]
    seq=fonte+resp
    assert len(seq)<=256, (q,len(seq))
    x=seq[:-1]
    y=[-100]*(len(fonte)-1)+seq[len(fonte):]
    assert len(x)==len(y)
    return x,y

def exemplo_doc(s):
    seq=[DOC]+cod(s)+[FIM]
    x=seq[:-1]; y=seq[1:]
    return x,y

def lote(bs=16, replay_p=.20):
    ex=[]
    for _ in range(bs):
        if random.random()<replay_p:
            ex.append(exemplo_doc(random.choice(REPLAY)))
        else:
            z=random.choice(TREINO)
            ex.append(exemplo_chat(z['q'],z['a']))
    L=max(len(x) for x,y in ex)
    X=torch.full((bs,L),PAD,dtype=torch.long,device=DEVICE)
    Y=torch.full((bs,L),-100,dtype=torch.long,device=DEVICE)
    for i,(x,y) in enumerate(ex):
        X[i,:len(x)]=torch.tensor(x,device=DEVICE)
        Y[i,:len(y)]=torch.tensor(y,device=DEVICE)
    return X,Y

@torch.no_grad()
def responder(pergunta,max_tokens=64):
    modelo.eval()
    fonte=fonte_dialogo(tok,pergunta,[],modelo.config.contexto)
    ids,terminou=modelo.gerar(fonte,FIM,max_tokens=max_tokens,temperatura=0,top_k=0,semente=SEED,proibidos=PROIB)
    return tok.decode(ids).strip(),terminou

HOLD=[
 ('Mesmo que um copo seja feito de barro, que tipo de coisa ele continua sendo?', [['recipiente']]),
 ('Junte em uma resposta o que um telescópio é e o que ele faz.', [['instrumento'],['observar','luz','distante']]),
 ('Uma hipótese ainda sem confirmação pode ser apresentada como certeza?', [['não'],['hipótese','possibilidade','provis']]),
 ('Se eu corrigi um nome que estava errado, qual versão deve orientar o restante da conversa?', [['corrig','nova','atualiz'],['anterior','errad','substit']]),
 ('De que maneira contexto e memória se complementam numa conversa?', [['contexto'],['memória'],['interpret','recuper']]),
 ('Um termômetro digital e um de líquido pertencem ao mesmo conceito geral de quê?', [['termômetro','instrumento'],['temperatura','medição']]),
 ('Explique por que uma garrafa de metal ainda pode ser chamada de garrafa.', [['garrafa'],['recipiente'],['material','metal','varia']]),
 ('Qual a diferença funcional entre uma pergunta e uma resposta?', [['pergunta'],['resposta'],['solic','fornec']]),
 ('Diga em uma frase o que é um microscópio e para que serve.', [['microscópio'],['instrumento'],['observar','ampliar','pequena']]),
 ('Se a água congela, ela necessariamente deixa de ser água?', [['não'],['água'],['estado','sólid','muda']]),
 ('O que significa dizer que uma correção atualiza o estado da conversa?', [['substit','troca','atualiz'],['informação'],['anterior']]),
 ('Como categoria e função ajudam a explicar um objeto?', [['categoria'],['função'],['é','faz','serve']]),
]

def norm(s):
    return re.sub(r'\s+',' ',s.casefold()).strip()

def score(resp,grupos):
    n=norm(resp)
    return all(any(op in n for op in grupo) for grupo in grupos)

@torch.no_grad()
def avaliar_hold():
    rows=[]; ok=0
    for q,g in HOLD:
        r,term=responder(q)
        ac=term and score(r,g)
        rows.append({'pergunta':q,'resposta':r,'terminou':term,'ok':ac})
        ok+=int(ac)
    return {'acertos':ok,'total':len(HOLD),'taxa':ok/len(HOLD),'casos':rows}

@torch.no_grad()
def ce_preservacao():
    modelo.eval(); perda=0.; nt=0
    for s in REPLAY:
        x,y=exemplo_doc(s)
        X=torch.tensor([x],device=DEVICE); Y=torch.tensor([y],device=DEVICE)
        log,_=modelo(X)
        perda+=float(torch.nn.functional.cross_entropy(log.reshape(-1,8192),Y.reshape(-1),reduction='sum'))
        nt+=Y.numel()
    return perda/nt

BASELINE={'holdout':avaliar_hold(),'preservacao_ce':ce_preservacao()}
print('ANTES - taxa:',BASELINE['holdout']['taxa'],'preservacao_ce:',BASELINE['preservacao_ce'])
for c in BASELINE['holdout']['casos'][:4]:
    print('\nQ:',c['pergunta'],'\nA:',c['resposta'],'\nOK:',c['ok'])


## TREINO REAL — composição semântica

Somente **blocos 6–7 + normalização final** são ajustados, com learning rate baixo. Os blocos 0–5 e embeddings permanecem congelados. Isso reduz o risco de apagar representações já aprendidas.

O checkpoint parcial é salvo no Drive a cada 300 passos.


In [ ]:
for p in modelo.parameters():
    p.requires_grad=False
for b in modelo.blocos[6:]:
    for p in b.parameters():
        p.requires_grad=True
for p in modelo.norm.parameters():
    p.requires_grad=True

pars=[p for p in modelo.parameters() if p.requires_grad]
print('Parâmetros treináveis:',sum(p.numel() for p in pars))

STEPS=1800
LR=7e-6
BS=16
opt=torch.optim.AdamW(pars,lr=LR,weight_decay=.01)
scaler=torch.amp.GradScaler('cuda')
OUTROOT=ROOT/'escola_composicional_real'
OUTROOT.mkdir(parents=True,exist_ok=True)
PARCIAL=OUTROOT/'pesos_parcial.pt'

LOG=[]
ini=time.time()
for step in range(1,STEPS+1):
    modelo.train()
    for b in modelo.blocos[:6]: b.eval()
    X,Y=lote(BS,replay_p=.20)
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda',dtype=torch.float16):
        logits,_=modelo(X)
        loss=torch.nn.functional.cross_entropy(logits.reshape(-1,8192),Y.reshape(-1),ignore_index=-100)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    grad=float(torch.nn.utils.clip_grad_norm_(pars,1.0))
    scaler.step(opt); scaler.update()

    if step==1 or step%50==0:
        r={'step':step,'loss':float(loss.detach()),'grad':grad,'seg':round(time.time()-ini,1)}
        LOG.append(r); print(r,flush=True)

    if step%300==0:
        parcial={
          'versao':estado['versao'],
          'modelo':{k:v.detach().cpu() for k,v in modelo.state_dict().items()},
          'config':estado['config'],
          'passo':int(estado.get('passo',0))+step,
          'execucao':{**estado.get('execucao',{}),
            'escola_composicional':{'step':step,'ativo_no_chat':False}}
        }
        torch.save(parcial,PARCIAL)
        print('💾 parcial:',PARCIAL,flush=True)

print('✅ treino terminou em',round(time.time()-ini,1),'s')


In [ ]:
FINAL_H=avaliar_hold()
FINAL_CE=ce_preservacao()
FINAL={'holdout':FINAL_H,'preservacao_ce':FINAL_CE}
REL={
 'holdout_gain':FINAL_H['taxa']-BASELINE['holdout']['taxa'],
 'preservacao_ce_ratio':FINAL_CE/BASELINE['preservacao_ce']
}

stamp=time.strftime('%Y%m%d_%H%M%S')
OUT=OUTROOT/f'crivo_composicional_{stamp}'
OUT.mkdir(parents=True)

novo={
 'versao':estado['versao'],
 'modelo':{k:v.detach().cpu() for k,v in modelo.state_dict().items()},
 'config':estado['config'],
 'passo':int(estado.get('passo',0))+STEPS,
 'execucao':{**estado.get('execucao',{}),
   'escola_composicional':{
      'origem':BASE.name,
      'intervencao':'composicao_semantica_em_respostas_inteiras',
      'steps':STEPS,'lr':LR,'batch':BS,
      'blocos_treinaveis':[6,7],
      'replay':0.20,
      'ativo_no_chat':False
   }}
}
torch.save(novo,OUT/'pesos.pt')
shutil.copy2(BASE/'tokenizer.json',OUT/'tokenizer.json')
ponte=BASE/'ponte_binding_v7_nao_revalidada.pt'
if ponte.exists():
    shutil.copy2(ponte,OUT/'ponte_binding_v7_nao_revalidada.pt')

meta={
 'ativo_no_chat':False,
 'promovido':False,
 'ponte_v7_requer_revalidacao':True,
 'baseline':BASELINE,
 'final':FINAL,
 'relativo':REL,
 'log':LOG
}
(OUT/'metricas.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
sha=hashlib.sha256((OUT/'pesos.pt').read_bytes()).hexdigest()

print('\n===== RESULTADO =====')
print('ANTES:',BASELINE['holdout']['acertos'],'/',BASELINE['holdout']['total'],
      'taxa=',round(BASELINE['holdout']['taxa'],4),
      'pres_ce=',round(BASELINE['preservacao_ce'],4))
print('DEPOIS:',FINAL_H['acertos'],'/',FINAL_H['total'],
      'taxa=',round(FINAL_H['taxa'],4),
      'pres_ce=',round(FINAL_CE,4))
print('MUDANÇA:',REL)
print('✅ CHECKPOINT FINAL:',OUT)
print('SHA256:',sha)
print('ativo_no_chat=false')

print('\n--- RESPOSTAS FINAIS ---')
for c in FINAL_H['casos']:
    print('\nQ:',c['pergunta'])
    print('A:',c['resposta'])
    print('OK:',c['ok'])
